# AI 抗性排行

AI Resistance Rankings — Which occupations are safest from automation?

评分色阶：红色(低分0) → 黄色(中等5) → 绿色(高分10)

In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.sans-serif'] = ['SimHei', 'Heiti TC', 'Arial Unicode MS', 'sans-serif']
matplotlib.rcParams['axes.unicode_minus'] = False

from pathlib import Path

csv_dir = Path('../data/csv')
all_files = sorted(csv_dir.glob('*.csv'))
print(f"Loading {len(all_files)} CSV files ...")

dfs = []
for f in all_files:
    tmp = pd.read_csv(f)
    dfs.append(tmp)
    print(f"  {f.name}: {len(tmp)} rows")

df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(df)} rows, {df['sub_category'].nunique()} occupations, "
      f"{df['country_or_region'].nunique()} countries/regions")


In [ ]:
# Key columns for AI analysis
base_cols = ['sub_category', 'sub_category_en', 'country_or_region', 'major_category', 'major_code', 'region']
ai_cols = ['ai_resistance', 'ai_timeline', 'value_added', 'growth_coeff',
           'skill_versatility', 'autonomy', 'physical_demand', 'composite_index']
display_cols = base_cols + ai_cols


## Top 50 Most AI-Resistant Occupations

In [ ]:
top50_resist = df.nlargest(50, 'ai_resistance')[display_cols].reset_index(drop=True)
top50_resist.index = top50_resist.index + 1
top50_resist.index.name = 'Rank'

score_gradient = [c for c in ai_cols if c not in ('ai_timeline',)]
styled = top50_resist.style \
    .background_gradient(subset=score_gradient, cmap='RdYlGn', vmin=0, vmax=10) \
    .set_properties(**{'font-size': '11px'})
styled


## Top 50 Most AI-Vulnerable Occupations (Lowest AI Resistance)

In [ ]:
top50_vuln = df.nsmallest(50, 'ai_resistance')[display_cols].reset_index(drop=True)
top50_vuln.index = top50_vuln.index + 1
top50_vuln.index.name = 'Rank'

styled = top50_vuln.style \
    .background_gradient(subset=score_gradient, cmap='RdYlGn', vmin=0, vmax=10) \
    .set_properties(**{'font-size': '11px'})
styled


## AI Resistance by Major Category

Mean AI resistance score and distribution summary per category.

In [ ]:
cat_ai = df.groupby('major_category')['ai_resistance'].agg(['mean', 'median', 'std', 'min', 'max']).round(2)
cat_ai = cat_ai.sort_values('mean', ascending=False)
cat_ai.columns = ['Mean', 'Median', 'Std', 'Min', 'Max']

styled = cat_ai.style \
    .background_gradient(subset=['Mean', 'Median'], cmap='RdYlGn', vmin=0, vmax=10) \
    .set_properties(**{'font-size': '12px'})
styled


In [ ]:
print("=== AI Resistance Distribution by Major Category ===\n")
for cat in cat_ai.index:
    row = cat_ai.loc[cat]
    q1 = df[df['major_category'] == cat]['ai_resistance'].quantile(0.25)
    q3 = df[df['major_category'] == cat]['ai_resistance'].quantile(0.75)
    print(f"{cat}:")
    print(f"  Mean={row['Mean']:.2f}, Median={row['Median']:.2f}, Std={row['Std']:.2f}")
    print(f"  Range=[{row['Min']:.1f}, {row['Max']:.1f}], IQR=[{q1:.1f}, {q3:.1f}]")
    print()


## AI Timeline Distribution

When will AI significantly impact each occupation?

In [ ]:
timeline_counts = df['ai_timeline'].value_counts().sort_index()
print("AI Impact Timeline Distribution:\n")
print(timeline_counts.to_string())
print(f"\nTotal: {timeline_counts.sum()}")

fig, ax = plt.subplots(figsize=(10, 5))
timeline_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('AI Impact Timeline Distribution', fontsize=14)
ax.set_xlabel('Timeline')
ax.set_ylabel('Number of Occupations')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


## AI Resistance vs Value Added

Are AI-safe jobs also well-paid? Scatter plot of ai_resistance vs value_added.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

categories = df['major_category'].unique()
colors = plt.cm.tab20(range(len(categories)))

for cat, color in zip(sorted(categories), colors):
    sub = df[df['major_category'] == cat]
    ax.scatter(sub['ai_resistance'], sub['value_added'],
               alpha=0.4, s=15, label=cat, color=color)

ax.set_xlabel('AI Resistance', fontsize=12)
ax.set_ylabel('Value Added', fontsize=12)
ax.set_title('AI Resistance vs Value Added by Category', fontsize=14)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axhline(y=5, color='gray', linestyle='--', alpha=0.3)
ax.axvline(x=5, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

# Correlation
corr = df[['ai_resistance', 'value_added']].corr().iloc[0, 1]
print(f"\nCorrelation between AI Resistance and Value Added: {corr:.3f}")
